In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, GRU, SimpleRNN, Bidirectional, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.optimizers import Adam

In [ ]:
import os

FILE_NAME = 'IMDB-Dataset.csv'
DRIVE_PATH = '/content/drive/MyDrive/AI projects/sentiment analysis/RNN,LSTM/' + FILE_NAME

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
if os.path.exists(FILE_NAME):
    print("Loading from local file")
    data = pd.read_csv(FILE_NAME)
else:
    print("Loading from Google Drive")
    data = pd.read_csv(DRIVE_PATH)

In [ ]:

data.head()

In [ ]:

data.columns

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
print(data.sentiment.value_counts())

In [ ]:
data.sentiment = [ 1 if each == "positive" else 0 for each in data.sentiment]

In [ ]:
data.head()

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
def clean_review(text):
    text = re.sub(r'<.*?>', ' ', text)        # remove html tags like <br />
    text = re.sub(r'[^a-zA-Z]', ' ', text)   # keep only letters
    text = text.lower()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

data['review'] = data['review'].apply(clean_review)

In [ ]:
sentiment = data['sentiment'].values
sentiment

In [ ]:
reviews = data['review']

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(reviews,sentiment,test_size = 0.2, random_state = 42)

<h2> Creating a Dictionary </h2>


In [ ]:
tokenizer = Tokenizer(num_words = 15000)
tokenizer.fit_on_texts(x_train)

<h2> Bring the comments to the same size </h2>

In [ ]:
x_train_tokens = tokenizer.texts_to_sequences(x_train)
x_test_tokens = tokenizer.texts_to_sequences(x_test)

In [ ]:
num_tokens = [len(tokens) for tokens in x_train_tokens]
num_tokens = np.array(num_tokens)

In [ ]:
max_tokens = np.mean(num_tokens) + 2 * np.std(num_tokens)
max_tokens = int(max_tokens)
max_tokens

In [ ]:
np.sum(num_tokens < max_tokens) / len(num_tokens)

In [ ]:
x_train_pad = pad_sequences(x_train_tokens, maxlen=max_tokens)
x_test_pad = pad_sequences(x_test_tokens, maxlen=max_tokens)

In [ ]:
x_train_pad.shape

In [ ]:
idx = tokenizer.word_index
inverse_map = dict(zip(idx.values(), idx.keys()))

def return_to_sentence(tokens):
    words = [inverse_map[token] for token in tokens if token!=0]
    text = ' '.join(words)
    return text

In [ ]:

print(return_to_sentence(x_train_pad[9]))

In [ ]:
print(x_train_pad[9])

In [ ]:
model = Sequential()

embedding_size = 50

model.add(Input(shape=(max_tokens,)))
model.add(Embedding(input_dim=15000,output_dim=embedding_size,name='embedding_layer'))

model.add(LSTM(units=16, return_sequences=True))
model.add(Dropout(0.1))

model.add(LSTM(units=8, return_sequences=True))
model.add(Dropout(0.1))

model.add(LSTM(units=4))
model.add(Dropout(0.1))

model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=1e-3),metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
history = model.fit(x_train_pad, y_train, validation_split=0.3, epochs=5, batch_size=1000, shuffle=True, verbose = 1)

In [ ]:
result = model.evaluate(x_test_pad, y_test)

In [ ]:
plt.figure()
plt.plot(history.history["accuracy"], label = "Train")
plt.plot(history.history["val_accuracy"], label = "Validation")
plt.title("Accuracy")
plt.ylabel("Acc")
plt.xlabel("epochs")
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(history.history["loss"], label = "Train")
plt.plot(history.history["val_loss"], label = "Validation")
plt.title("Loss")
plt.ylabel("Loss")
plt.xlabel("epochs")
plt.legend()
plt.show()

In [ ]:
rnn = Sequential()

embedding_size = 50

rnn.add(Input(shape=(max_tokens,)))
rnn.add(Embedding(input_dim=15000,output_dim=embedding_size))

rnn.add(SimpleRNN(units=16, return_sequences=True))
rnn.add(Dropout(0.1))

rnn.add(SimpleRNN(units=8, return_sequences=True))
rnn.add(Dropout(0.1))

rnn.add(SimpleRNN(units=4))
rnn.add(Dropout(0.1))

rnn.add(Dense(1, activation='sigmoid'))

rnn.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=1e-3),metrics=['accuracy'])

In [ ]:
rnn.summary()

In [ ]:
rnnhistory = rnn.fit(x_train_pad, y_train, validation_split=0.3, epochs=5, batch_size=1000, shuffle=True, verbose = 1)

In [ ]:
result = rnn.evaluate(x_test_pad, y_test)

In [ ]:
plt.figure()
plt.plot(rnnhistory.history["accuracy"], label = "Train")
plt.plot(rnnhistory.history["val_accuracy"], label = "Validation")
plt.title("Accuracy")
plt.ylabel("Acc")
plt.xlabel("epochs")
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(rnnhistory.history["loss"], label = "Train")
plt.plot(rnnhistory.history["val_loss"], label = "Validation")
plt.title("Loss")
plt.ylabel("Loss")
plt.xlabel("epochs")
plt.legend()
plt.show()

In [ ]:
gru = Sequential()

embedding_size = 50

gru.add(Input(shape=(max_tokens,)))
gru.add(Embedding(input_dim=15000,output_dim=embedding_size))

gru.add(GRU(units=16, return_sequences=True))
gru.add(Dropout(0.1))

gru.add(GRU(units=8, return_sequences=True))
gru.add(Dropout(0.1))

gru.add(GRU(units=4))
gru.add(Dropout(0.1))

gru.add(Dense(1, activation='sigmoid'))

gru.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=1e-3),metrics=['accuracy'])

In [ ]:
gru.summary()

In [ ]:
gruhistory = gru.fit(x_train_pad, y_train, validation_split=0.3, epochs=5, batch_size=1000, shuffle=True)

In [ ]:
result = gru.evaluate(x_test_pad, y_test)

In [ ]:
plt.figure()
plt.plot(gruhistory.history["accuracy"], label = "Train")
plt.plot(gruhistory.history["val_accuracy"], label = "Validation")
plt.title("Accuracy")
plt.ylabel("Acc")
plt.xlabel("epochs")
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(gruhistory.history["loss"], label = "Train")
plt.plot(gruhistory.history["val_loss"], label = "Validation")
plt.title("Loss")
plt.ylabel("Loss")
plt.xlabel("epochs")
plt.legend()
plt.show()

In [ ]:
bigru = Sequential()

embedding_size = 50

bigru.add(Input(shape=(max_tokens,)))
bigru.add(Embedding(input_dim=15000,output_dim=embedding_size))

bigru.add(Bidirectional(GRU(units=16, return_sequences=True)))
bigru.add(Dropout(0.1))

bigru.add(Bidirectional(GRU(units=8, return_sequences=True)))
bigru.add(Dropout(0.1))

bigru.add(Bidirectional(GRU(units=4)))
bigru.add(Dropout(0.1))

bigru.add(Dense(1, activation='sigmoid'))

bigru.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=1e-3),metrics=['accuracy'])

In [ ]:
bigru.summary()

In [ ]:
bigruhistory = bigru.fit(x_train_pad, y_train, validation_split=0.3, epochs=5, batch_size=1000, shuffle=True)

In [ ]:
result = bigru.evaluate(x_test_pad, y_test)

In [ ]:
plt.figure()
plt.plot(bigruhistory.history["accuracy"], label = "Train")
plt.plot(bigruhistory.history["val_accuracy"], label = "Validation")
plt.title("Accuracy")
plt.ylabel("Acc")
plt.xlabel("epochs")
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(bigruhistory.history["loss"], label = "Train")
plt.plot(bigruhistory.history["val_loss"], label = "Validation")
plt.title("Loss")
plt.ylabel("Loss")
plt.xlabel("epochs")
plt.legend()
plt.show()

In [ ]:
bilstm = Sequential()

embedding_size = 50

bilstm.add(Input(shape=(max_tokens,)))
bilstm.add(Embedding(input_dim=15000,output_dim=embedding_size))

bilstm.add(Bidirectional(LSTM(units=16, return_sequences=True)))
bilstm.add(Dropout(0.1))

bilstm.add(Bidirectional(LSTM(units=8, return_sequences=True)))
bilstm.add(Dropout(0.1))

bilstm.add(Bidirectional(LSTM(units=4)))
bilstm.add(Dropout(0.1))

bilstm.add(Dense(1, activation='sigmoid'))

bilstm.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=1e-3),metrics=['accuracy'])
bilstm.summary()

In [ ]:
bi_lhistory = bilstm.fit(x_train_pad, y_train, validation_split=0.3, epochs=5, batch_size=1000, shuffle=True)

In [ ]:
result = bilstm.evaluate(x_test_pad, y_test)

In [ ]:
plt.figure()
plt.plot(bi_lhistory.history["accuracy"], label = "Train")
plt.plot(bi_lhistory.history["val_accuracy"], label = "Validation")
plt.title("Accuracy")
plt.ylabel("Acc")
plt.xlabel("epochs")
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(bi_lhistory.history["loss"], label = "Train")
plt.plot(bi_lhistory.history["val_loss"], label = "Validation")
plt.title("Loss")
plt.ylabel("Loss")
plt.xlabel("epochs")
plt.legend()
plt.show()